In [10]:
# Run this in a TERMINAL (not this notebook cell) -- it takes 1-3 minutes:
#
#   minikube start --cpus 4 --memory 4096 --driver=docker
#
# Then verify from here:
import subprocess

def sh(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr)
    return result

sh("kubectl cluster-info")
sh("kubectl get nodes -o wide")

Kubernetes control plane is running at https://192.168.49.2:8443
CoreDNS is running at https://192.168.49.2:8443/api/v1/namespaces/kube-system/services/kube-dns:dns/proxy

To further debug and diagnose cluster problems, use 'kubectl cluster-info dump'.

NAME       STATUS   ROLES           AGE   VERSION   INTERNAL-IP    EXTERNAL-IP   OS-IMAGE                         KERNEL-VERSION     CONTAINER-RUNTIME
minikube   Ready    control-plane   10d   v1.35.1   192.168.49.2   <none>        Debian GNU/Linux 12 (bookworm)   7.0.0-30-generic   docker://29.2.1



CompletedProcess(args='kubectl get nodes -o wide', returncode=0, stdout='NAME       STATUS   ROLES           AGE   VERSION   INTERNAL-IP    EXTERNAL-IP   OS-IMAGE                         KERNEL-VERSION     CONTAINER-RUNTIME\nminikube   Ready    control-plane   10d   v1.35.1   192.168.49.2   <none>        Debian GNU/Linux 12 (bookworm)   7.0.0-30-generic   docker://29.2.1\n', stderr='')

In [11]:
import subprocess, os
os.makedirs("data", exist_ok=True)
subprocess.run(["python3", "dataset2.py"], check=True)
print("Dataset ready.")

Successfully generated 8 CSV shards in the 'data/' directory.
Dataset ready.


In [12]:
%%writefile email_validity_checker.py
def classify(email):
    if not isinstance(email, str) or not email.strip():
        return "invalid"
    if not email or "@" not in email or "." not in email or " " in email:
        return 'Invalid'
    return 'Valid'

import json
import os
import pandas as pd

shard_index = int(os.environ.get("JOB_COMPLETION_INDEX", "0"))
node_name = os.environ.get("NODE_NAME", "unknown")
pod_name = os.environ.get("POD_NAME", "unknown")

file_path = f"/app/data/signup_shard_{shard_index}.csv"
df = pd.read_csv(file_path)

invalid_count = 0

for idx,row in df.iterrows():
    result = classify(row['email'])
    if(result == 'Invalid'):
        invalid_count += 1

results = {'completion_index' : shard_index, 'invalid_counts' : invalid_count}

print(f"RESULT_JSON:{json.dumps(results)}")


Overwriting email_validity_checker.py


In [13]:
%%writefile collect_results.py
"""
collect_results.py — AI Operations (AIOps), Module 3 Lecture 2a
Collects RESULT_JSON lines from every pod of a completed Job via the Kubernetes API.
"""
import argparse, json, re, sys
import pandas as pd
from kubernetes import client, config

RESULT_LINE_RE = re.compile(r"RESULT_JSON:(\{.*\})")


def load_kube_config():
    try:
        config.load_kube_config()
    except Exception:
        config.load_incluster_config()


def collect(job_name, namespace="default"):
    load_kube_config()
    v1 = client.CoreV1Api()
    pods = v1.list_namespaced_pod(namespace=namespace, label_selector=f"job-name={job_name}")
    if not pods.items:
        print(f"No pods found for job '{job_name}'.", file=sys.stderr)
        return pd.DataFrame()

    rows = []
    for pod in pods.items:
        pod_name = pod.metadata.name
        try:
            logs = v1.read_namespaced_pod_log(name=pod_name, namespace=namespace)
        except client.exceptions.ApiException as e:
            print(f"  Could not read logs for {pod_name}: {e.reason}", file=sys.stderr)
            continue
        match = RESULT_LINE_RE.search(logs)
        if not match:
            print(f"  No RESULT_JSON line in {pod_name} yet.", file=sys.stderr)
            continue
        result = json.loads(match.group(1))
        result["k8s_pod_phase"] = pod.status.phase
        rows.append(result)

    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values("completion_index").reset_index(drop=True)
    return df



if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--job-name", required=True)
    parser.add_argument("--namespace", default="default")
    parser.add_argument("--out", default=None)
    args = parser.parse_args()

    df = collect(args.job_name, args.namespace)
    if df.empty:
        print("No results collected.")
    else:
        pd.set_option("display.width", 120)
        print(df.to_string(index=False))
        print("\nNodes that participated:", sorted(df["node_name"].unique()))
        if args.out:
            df.to_csv(args.out, index=False)


Overwriting collect_results.py


In [ ]:
%%writefile job-multi-node.yaml
apiVersion: batch/v1
kind: Job
metadata:
  name: email-verifier-job
  labels:
    app: email-verifier
spec:
  completions: 8
  parallelism: 4
  completionMode: Indexed
  backoffLimit: 4
  activeDeadlineSeconds: 1800
  template:
    metadata:
      labels:
        app: email-verifier
    spec:
      restartPolicy: Never
      topologySpreadConstraints:
      - maxSkew: 1
        topologyKey: kubernetes.io/hostname
        whenUnsatisfiable: DoNotSchedule
        labelSelector:
          matchLabels:
            app: email-verifier
      containers:
      - name: worker
        image: video_test_email_verifier:latest
        imagePullPolicy: Never
        env:
        - name: POD_NAME
          valueFrom:
            fieldRef:
              fieldPath: metadata.name
        - name: NODE_NAME
          valueFrom:
            fieldRef:
              fieldPath: spec.nodeName
        resources:
          requests:
            cpu: "500m"
            memory: "128Mi"
          limits:
            cpu: "1000m"
            memory: "256Mi"


Overwriting job-multi-node.yaml


In [17]:
%pip install kubernetes
from collect_results import collect

results_df = collect("email-verifier-job")
results_df

Note: you may need to restart the kernel to use updated packages.


,completion_index,invalid_counts,k8s_pod_phase
0,0,12,Succeeded
1,1,16,Succeeded
2,2,15,Succeeded
3,3,14,Succeeded
4,4,18,Succeeded
5,5,15,Succeeded
6,6,20,Succeeded
7,7,19,Succeeded
